# Destilar la voz **Alex** (Kokoro) a Piper — 2 celdas
Genera una voz Piper con el timbre de `em_alex`. Sin grabar nada.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
Después: **celda 1** (prepara todo, ~15 min) y **celda 2** (entrena, 2-4 h).

In [ ]:
#@title 1. PREPARAR TODO (Kokoro genera el dataset + instala Piper) — ~15 min
import torch, os
assert torch.cuda.is_available(), "Activá T4: Entorno de ejecución -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

!pip install -q kokoro-onnx==0.5.0
!wget -q -nc -O /content/kokoro.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"
!wget -q -nc -O /content/voices.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"
!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"

import wave, numpy as np
from kokoro_onnx import Kokoro
frases = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]
print(f"{len(frases)} frases. Generando audio con la voz Alex...")
kokoro = Kokoro("/content/kokoro.onnx", "/content/voices.bin")
os.makedirs("/content/dataset/wavs", exist_ok=True)
rows, total = [], 0.0
for i, f in enumerate(frases):
    try: s, r = kokoro.create(f, voice="em_alex", speed=1.0, lang="es")
    except Exception: continue
    idx = np.linspace(0, len(s)-1, int(len(s)*22050/r))
    d = np.clip(np.interp(idx, np.arange(len(s)), s)*32767, -32768, 32767).astype(np.int16)
    if not 1.0 <= len(d)/22050 <= 20.0: continue
    n = f"f{i:05d}"
    with wave.open(f"/content/dataset/wavs/{n}.wav","wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
    rows.append(f"{n}|{f}"); total += len(d)/22050
    if len(rows)%200==0: print(f"  {len(rows)} frases...")
open("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")
print(f"Dataset: {len(rows)} clips, {total/60:.1f} min. Instalando Piper (unos minutos)...")

%cd /content
!git clone -q https://github.com/rhasspy/piper.git
%cd /content/piper/src/python
!pip install -q -e . "pytorch-lightning~=1.9" espeak-phonemizer librosa "numpy<2"
!apt-get install -yq espeak-ng > /dev/null
!bash build_monotonic_align.sh > /dev/null 2>&1
!python -m piper_train.preprocess --language es --input-dir /content/dataset --output-dir /content/train_out --dataset-format ljspeech --single-speaker --sample-rate 22050
!wget -q -nc -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"
print("\n=== LISTO. Ahora corré la celda 2 para entrenar. ===")

In [ ]:
#@title 2. ENTRENAR Y DESCARGAR — 2-4 h (podés cortarla y reejecutarla para exportar lo entrenado)
import glob, shutil, os
%cd /content/piper/src/python
ck = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)
if not ck:
    !python -m piper_train --dataset-dir /content/train_out --accelerator gpu --devices 1 --batch-size 16 --validation-split 0.0 --num-test-examples 0 --max_epochs 3219 --resume_from_checkpoint /content/base.ckpt --checkpoint-epochs 5 --precision 32 --quality medium
    ck = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)
assert ck, "No hay checkpoint todavía: dejá entrenar unos minutos y reejecutá esta celda."
!python -m piper_train.export_onnx "{ck[-1]}" /content/es_alex_kokoro.onnx
shutil.copy("/content/train_out/config.json", "/content/es_alex_kokoro.onnx.json")
from google.colab import files
files.download("/content/es_alex_kokoro.onnx")
files.download("/content/es_alex_kokoro.onnx.json")
print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'es_alex_kokoro' en Configuración")